In [4]:
import os
import yaml
import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
import pandas as pd 


with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

api_user = config["credentials"]["username"]
api_pass = config["credentials"]["password"]
auth_credentials = HTTPBasicAuth(api_user, api_pass)

target_group_id = int(input("Enter group id: "))
schedule_url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={target_group_id}&okres=2"

web_response = requests.get(schedule_url, auth=auth_credentials)
web_response.encoding = 'utf-8'
print(f"Server response status: {web_response.status_code}")

dom = BeautifulSoup(web_response.text, "html.parser")
parsed_group_name = dom.select_one("div.grupa").get_text(strip=True)
print(f"Group identity: {parsed_group_name}")

table_tag = dom.select_one("table")
with open("temp.html", "w", encoding="utf-8") as temp_file:
    temp_file.write(table_tag.prettify())

classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

allowed_types = ["ćwiczenia", "wykład", "egzamin"]
classes = classes.loc[classes["Typ"].isin(allowed_types)]

time_parts = classes["Dzień, godzina"].str.split(" ", expand=True)
classes["Day"] = time_parts[0]
classes["Start time"] = time_parts[1]
classes["End time"] = time_parts[3]
classes["Duration"] = time_parts[4].str.extract(r"(\d+)").astype(int)

classes = classes.drop(columns=["Dzień, godzina"])
classes["Sala"] = classes["Sala"].str.replace(r'Win.*', '', regex=True)

if not os.path.exists("schedules"):
    os.mkdir("schedules")
classes.to_csv(f"schedules/{parsed_group_name}.csv")

Server response status: 200
Group identity: ZICSS1-1211
